In [1]:
%pip install composable

Note: you may need to restart the kernel to use updated packages.


In [2]:
import composable.records as rec

In [122]:
# Standard imports
import polars as pl
import polars.selectors as cs
import seaborn as sns
import numpy as np

from sklearn.preprocessing import StandardScaler

# Pipeline
from sklearn.pipeline import Pipeline


# Preprocessing stuff
from sklearn.preprocessing import LabelEncoder

# Model selection stuff
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV

# Classic classifiers
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier


from sklearn.multiclass import OneVsRestClassifier, OneVsOneClassifier


# Metrics to use on the test set
# metric(y_test, y_predict)
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score, roc_auc_score




# Trees and Forests in `sklearn`

In this notebook, we will cover the basics of classification, including:

1. The `DecisionTreeClassifier` and `RandomForestClassifier`.
2. Exploring tuning parameters for each model.
3. Performing a grid search.
4. Additional features of Random Forests.
5. Multiclass problems.

## Classification Example: Kyphosis Data Set

**Problem Statement:**
Given a DataSet of 81 patients who have undergone a spinal surgery for a deformation and the data if the condition recurred, Build a claddification Model to predict whether a patient being admitted for the surgery has chance for recurrence. This model will help the surgeons to plan appropriate level of treatment to prevent recurrence.

**Data Set Description :**
The kyphosis data frame has 81 rows and 4 columns. representing data on children who have had corrective spinal surgery

This data frame contains the following columns/Features:

1. *Kyphosis*: a factor with levels absent present indicating if a kyphosis (a type of deformation) was present after the operation.
2. *Age*: in months
3. *Number*: the number of vertebrae involved
4. *Start*: the number of the first (topmost) vertebra operated on.

**Research Question:** Can we predict whether if kyphosis will be present or absent after the surgery based on the age of the patient, number of vertebrae involved and the first vertebra operated on?

In [4]:
(kyphosis :=
 pl.read_csv('sample_data/kyphosis.csv')
   .drop('rownames')
)

Kyphosis,Age,Number,Start
str,i64,i64,i64
"""absent""",71,3,5
"""absent""",158,3,14
"""present""",128,4,5
"""absent""",2,5,1
"""absent""",1,4,15
…,…,…,…
"""present""",157,3,13
"""absent""",26,7,13
"""absent""",120,2,13


In [5]:
(X_kyphosis :=
 kyphosis
 .drop('Kyphosis')
 .to_pandas()
)

,Age,Number,Start
0,71,3,5
1,158,3,14
2,128,4,5
3,2,5,1
4,1,4,15
...,...,...,...
76,157,3,13
77,26,7,13
78,120,2,13
79,42,7,6


In [6]:
(y_kyphosis :=
 kyphosis
 .get_column('Kyphosis')
 .to_numpy()
 .ravel()
)

array(['absent', 'absent', 'present', 'absent', 'absent', 'absent',
       'absent', 'absent', 'absent', 'present', 'present', 'absent',
       'absent', 'absent', 'absent', 'absent', 'absent', 'absent',
       'absent', 'absent', 'absent', 'present', 'present', 'absent',
       'present', 'absent', 'absent', 'absent', 'absent', 'absent',
       'absent', 'absent', 'absent', 'absent', 'absent', 'absent',
       'absent', 'present', 'absent', 'present', 'present', 'absent',
       'absent', 'absent', 'absent', 'present', 'absent', 'absent',
       'present', 'absent', 'absent', 'absent', 'present', 'absent',
       'absent', 'absent', 'absent', 'present', 'absent', 'absent',
       'present', 'present', 'absent', 'absent', 'absent', 'absent',
       'absent', 'absent', 'absent', 'absent', 'absent', 'absent',
       'absent', 'absent', 'absent', 'absent', 'present', 'absent',
       'absent', 'present', 'absent'], dtype=object)

## Topic 1 - Tree and Forest models in `sklearn`

`sklearn` comes with both a tree and forest based classifier, which can be found in the `tree` and `ensemble` submodules, respectively.

In [7]:
(tree := DecisionTreeClassifier(class_weight='balanced')
)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the curre

In [8]:
(forest := RandomForestClassifier(class_weight='balanced')
)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

## Tuning parameters for trees and forests

First, let's inspect the tuning parameters for each model.

#### Exploring trees

In [9]:
?tree

Type:        DecisionTreeClassifier
String form: DecisionTreeClassifier(class_weight='balanced')
File:        c:\users\lr7273ow\appdata\local\anaconda3\envs\polars\lib\site-packages\sklearn\tree\_classes.py
Docstring:  
A decision tree classifier.

Read more in the :ref:`User Guide <tree>`.

Parameters
----------
criterion : {"gini", "entropy", "log_loss"}, default="gini"
    The function to measure the quality of a split. Supported criteria are
    "gini" for the Gini impurity and "log_loss" and "entropy" both for the
    Shannon information gain, see :ref:`tree_mathematical_formulation`.

splitter : {"best", "random"}, default="best"
    The strategy used to choose the split at each node. Supported
    strategies are "best" to choose the best split and "random" to choose
    the best random split.

max_depth : int, default=None
    The maximum depth of the tree. If None, then nodes are expanded until
    all leaves are pure or until all leaves contain less than
    min_samples_split 

In [10]:
tree.get_params()

{'ccp_alpha': 0.0,
 'class_weight': 'balanced',
 'criterion': 'gini',
 'max_depth': None,
 'max_features': None,
 'max_leaf_nodes': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'monotonic_cst': None,
 'random_state': None,
 'splitter': 'best'}

In [11]:
(tree_grid :=
 {'min_samples_leaf': [1,2,3],
  'min_samples_split': [2,3,4],
  'max_depth': [3,4,5],
 }
)

{'min_samples_leaf': [1, 2, 3],
 'min_samples_split': [2, 3, 4],
 'max_depth': [3, 4, 5]}

In [12]:
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [13]:
grid_search_tree = GridSearchCV(tree, tree_grid, cv = folds, scoring='roc_auc', n_jobs=-1, verbose=1)

grid_search_tree.fit(X_kyphosis, y_kyphosis)


Fitting 5 folds for each of 27 candidates, totalling 135 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",DecisionTreeC...ht='balanced')
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'max_depth': [3, 4, ...], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 3, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and param

In [14]:
?cross_validate

Signature:
cross_validate(
    estimator,
    X,
    y=None,
    *,
    groups=None,
    scoring=None,
    cv=None,
    n_jobs=None,
    verbose=0,
    params=None,
    pre_dispatch='2*n_jobs',
    return_train_score=False,
    return_estimator=False,
    return_indices=False,
    error_score=nan,
)
Docstring:
Evaluate metric(s) by cross-validation and also record fit/score times.

Read more in the :ref:`User Guide <multimetric_cross_validation>`.

Parameters
----------
estimator : estimator object implementing 'fit'
    The object to use to fit the data.

X : {array-like, sparse matrix} of shape (n_samples, n_features)
    The data to fit. Can be for example a list, or an array.

y : array-like of shape (n_samples,) or (n_samples, n_outputs), default=None
    The target variable to try to predict in the case of
    supervised learning.

groups : array-like of shape (n_samples,), default=None
    Group labels for the samples used while splitting the dataset into
    train/test set. Onl

In [15]:
my_scores = ['accuracy', 'balanced_accuracy', 'roc_auc']

(tree_scores :=
 cross_validate(grid_search_tree, X_kyphosis, y_kyphosis,
                cv = folds,
                scoring=my_scores,
                verbose=0,
                n_jobs=-1,
                )
 >> rec.subset([f'test_{m}' for m in my_scores])
 >> rec.map(np.mean)
)

{'test_accuracy': np.float64(0.7176470588235294),
 'test_balanced_accuracy': np.float64(0.6576923076923077),
 'test_roc_auc': np.float64(0.7096153846153845)}

#### Exploring forests

In [36]:
?forest

Type:        RandomForestClassifier
String form: RandomForestClassifier(class_weight='balanced')
File:        c:\users\lr7273ow\appdata\local\anaconda3\envs\polars\lib\site-packages\sklearn\ensemble\_forest.py
Docstring:  
A random forest classifier.

A random forest is a meta estimator that fits a number of decision tree
classifiers on various sub-samples of the dataset and uses averaging to
improve the predictive accuracy and control over-fitting.
Trees in the forest use the best split strategy, i.e. equivalent to passing
`splitter="best"` to the underlying :class:`~sklearn.tree.DecisionTreeClassifier`.
The sub-sample size is controlled with the `max_samples` parameter if
`bootstrap=True` (default), otherwise the whole dataset is used to build
each tree.

For a comparison between tree-based ensemble models see the example
:ref:`sphx_glr_auto_examples_ensemble_plot_forest_hist_grad_boosting_comparison.py`.

This estimator has native support for missing values (NaNs). During training,


In [16]:
forest.get_params()

{'bootstrap': True,
 'ccp_alpha': 0.0,
 'class_weight': 'balanced',
 'criterion': 'gini',
 'max_depth': None,
 'max_features': 'sqrt',
 'max_leaf_nodes': None,
 'max_samples': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'monotonic_cst': None,
 'n_estimators': 100,
 'n_jobs': None,
 'oob_score': False,
 'random_state': None,
 'verbose': 0,
 'warm_start': False}

In [17]:
(forest_grid :=
{'max_depth': [2,3,4,5],
 'min_samples_leaf': [1,2,3],
 'min_samples_split': [2,3,4],
 'n_estimators': [10, 100, 500],
 }
)

{'max_depth': [2, 3, 4, 5],
 'min_samples_leaf': [1, 2, 3],
 'min_samples_split': [2, 3, 4],
 'n_estimators': [10, 100, 500]}

In [18]:
grid_search_forest = GridSearchCV(forest, forest_grid, cv = folds, scoring='roc_auc', n_jobs=-1, verbose=1)

grid_search_forest.fit(X_kyphosis, y_kyphosis)

Fitting 5 folds for each of 108 candidates, totalling 540 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestC...ht='balanced')
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'max_depth': [2, 3, ...], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 3, ...], 'n_estimators': [10, 100, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computat

In [19]:
(forest_scores :=
 cross_validate(grid_search_tree, X_kyphosis, y_kyphosis,
                cv = folds,
                scoring=my_scores,
                verbose=0,
                n_jobs=-1,
                )
 >> rec.subset([f'test_{m}' for m in my_scores])
 >> rec.map(np.mean)
)


{'test_accuracy': np.float64(0.7176470588235294),
 'test_balanced_accuracy': np.float64(0.6576923076923077),
 'test_roc_auc': np.float64(0.6846153846153846)}

## Topic 3 - Extra random forest features

As you learned from your out of class videos, the fact that random forests uses bagging allows for extra functionality, including

1. A measure of variable importance, and
2. An out of the box objective measurement of model fit.

#### Variable importance

In [20]:
grid_search_forest.best_estimator_.feature_importances_ # Ugh! so unreadable!

array([0.36306302, 0.3125026 , 0.32443437])

In [21]:
(feat_imp :=
 pl.DataFrame({'columns':grid_search_forest.feature_names_in_,
               'importance':grid_search_forest.best_estimator_.feature_importances_},
             )
   .sort('importance', descending=True)
)


columns,importance
str,f64
"""Age""",0.363063
"""Start""",0.324434
"""Number""",0.312503


#### Out of bag error estimate

**Notes.**

1. You need to set `oob_score=True` when creating the forest to get this measure.
2. Computing the `oob_score` has some overhead, so it is best leave this off when performing a grid search.  Instead, refit the winning model with this flag after.

In [23]:
grid_search_forest.best_params_

{'max_depth': 3,
 'min_samples_leaf': 1,
 'min_samples_split': 3,
 'n_estimators': 10}

In [22]:
(winning_forest_w_oob_error :=
 RandomForestClassifier(oob_score=True,
                        
                       **grid_search_forest.best_params_,
                      )

)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",10
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",3
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",3
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y_t

In [24]:
winning_forest_w_oob_error.fit(X_kyphosis, y_kyphosis)

winning_forest_w_oob_error.oob_score_

0.7654320987654321

## Topic 4 - Multiclass classification

In [25]:
(olive_oil :=
 pl.read_csv("sample_data/OliveOils.csv")
).head(2)

Area.name,palmitic,palmitoleic,strearic,oleic,linoleic,eicosanoic,linolenic
str,i64,i64,i64,i64,i64,i64,i64
"""North-Apulia""",1075,75,226,7823,672,36,60
"""North-Apulia""",1088,73,224,7709,781,31,61


In [26]:
(X_oil :=
 olive_oil
 .drop('Area.name')
 .to_pandas()
).head(2)

,palmitic,palmitoleic,strearic,oleic,linoleic,eicosanoic,linolenic
0,1075,75,226,7823,672,36,60
1,1088,73,224,7709,781,31,61


In [27]:
(y_oil :=
 olive_oil
 .select('Area.name')
 .to_numpy()
 .ravel()
)[:3]

array(['North-Apulia', 'North-Apulia', 'North-Apulia'], dtype=object)

In [28]:
X_train_oil, X_test_oil, y_train_oil, y_test_oil = train_test_split(X_oil, y_oil, test_size=0.3, random_state=42, stratify=y_oil)

In [29]:
grid_search_tree_MC = GridSearchCV(tree, tree_grid, cv = folds, scoring='balanced_accuracy', n_jobs=-1, verbose=0)

grid_search_tree_MC.fit(X_train_oil, y_train_oil)
grid_search_tree_MC.score(X_test_oil, y_test_oil)

0.7979253089310016

In [30]:
grid_search_forest_MC = GridSearchCV(forest, forest_grid, cv = folds, scoring='balanced_accuracy', n_jobs=-1, verbose=0)

grid_search_forest_MC.fit(X_train_oil, y_train_oil)
grid_search_forest_MC.score(X_test_oil, y_test_oil)

0.8929514718888534

## <font color='red'> Exercise 1 </font>

Do some sleuthing to determine the impact of various tuning parameters for both trees and forests on the model flexibility and overfitting.

<font color='orange'>
A summary of your findings here.
</font>

In [39]:
?tree

Type:        DecisionTreeClassifier
String form: DecisionTreeClassifier(class_weight='balanced')
File:        c:\users\lr7273ow\appdata\local\anaconda3\envs\polars\lib\site-packages\sklearn\tree\_classes.py
Docstring:  
A decision tree classifier.

Read more in the :ref:`User Guide <tree>`.

Parameters
----------
criterion : {"gini", "entropy", "log_loss"}, default="gini"
    The function to measure the quality of a split. Supported criteria are
    "gini" for the Gini impurity and "log_loss" and "entropy" both for the
    Shannon information gain, see :ref:`tree_mathematical_formulation`.

splitter : {"best", "random"}, default="best"
    The strategy used to choose the split at each node. Supported
    strategies are "best" to choose the best split and "random" to choose
    the best random split.

max_depth : int, default=None
    The maximum depth of the tree. If None, then nodes are expanded until
    all leaves are pure or until all leaves contain less than
    min_samples_split 

In [37]:
tree.get_params()

{'ccp_alpha': 0.0,
 'class_weight': 'balanced',
 'criterion': 'gini',
 'max_depth': None,
 'max_features': None,
 'max_leaf_nodes': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'monotonic_cst': None,
 'random_state': None,
 'splitter': 'best'}

In [40]:
?forest

Type:        RandomForestClassifier
String form: RandomForestClassifier(class_weight='balanced')
File:        c:\users\lr7273ow\appdata\local\anaconda3\envs\polars\lib\site-packages\sklearn\ensemble\_forest.py
Docstring:  
A random forest classifier.

A random forest is a meta estimator that fits a number of decision tree
classifiers on various sub-samples of the dataset and uses averaging to
improve the predictive accuracy and control over-fitting.
Trees in the forest use the best split strategy, i.e. equivalent to passing
`splitter="best"` to the underlying :class:`~sklearn.tree.DecisionTreeClassifier`.
The sub-sample size is controlled with the `max_samples` parameter if
`bootstrap=True` (default), otherwise the whole dataset is used to build
each tree.

For a comparison between tree-based ensemble models see the example
:ref:`sphx_glr_auto_examples_ensemble_plot_forest_hist_grad_boosting_comparison.py`.

This estimator has native support for missing values (NaNs). During training,


 #### The impact of certain parameters for the tree classification model such as criterion, splitter and max_depth params. The ccp_alpha parameter is able to chop off branches that don't add predictive power uses the measure to the quality of a split, the min_samples_leaf is used to choose the split at each node, and the max_depth (which is the most important) deals with overfitting. 

#### The impact of certain parameters for the forest classification model such as n_estimator determines  the number of trees, the max features how many limits each tree can look for a split, and the bootstrap as the tree uses sub-samples which averages out the mean between trees.  

## <font color="red"> Exercise 2 </font>

1. Use a combined grid search to compare the performance of trees and forests on the Pima Indian diabetes data set.  
2. Validate the winning model by computing CV scores on the test set.
3. Use random forests to explore the feature importance.  If a forest wasn't the winning model, redo the grid search to find the best forest.
4. Comment on your findings.

In [44]:
(
    diabetes := 
 pl.read_csv('sample_data/diabetes_raw.csv')
)

Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Diabetes
i64,i64,i64,i64,i64,f64,f64,i64,str
6,148,72,35,0,33.6,0.627,50,"""Yes"""
1,85,66,29,0,26.6,0.351,31,"""No"""
8,183,64,0,0,23.3,0.672,32,"""Yes"""
1,89,66,23,94,28.1,0.167,21,"""No"""
0,137,40,35,168,43.1,2.288,33,"""Yes"""
…,…,…,…,…,…,…,…,…
10,101,76,48,180,32.9,0.171,63,"""No"""
2,122,70,27,0,36.8,0.34,27,"""No"""
5,121,72,23,112,26.2,0.245,30,"""No"""


In [ ]:
(X_diabetes :=
    diabetes
    .drop('Diabetes')
    .to_pandas()
 )

Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
i64,i64,i64,i64,i64,f64,f64,i64
6,148,72,35,0,33.6,0.627,50
1,85,66,29,0,26.6,0.351,31
8,183,64,0,0,23.3,0.672,32
1,89,66,23,94,28.1,0.167,21
0,137,40,35,168,43.1,2.288,33
…,…,…,…,…,…,…,…
10,101,76,48,180,32.9,0.171,63
2,122,70,27,0,36.8,0.34,27
5,121,72,23,112,26.2,0.245,30


In [ ]:
( y_diabetes :=
       diabetes 
       .select('Diabetes')
       .to_pandas()
)

Diabetes
str
"""Yes"""
"""No"""
"""Yes"""
"""No"""
"""Yes"""
…
"""No"""
"""No"""
"""No"""


In [65]:
X_train_diabetes, X_test_diabetes, y_train_diabetes, y_test_diabetes = train_test_split(X_diabetes,y_diabetes, test_size=0.3, random_state=42) 

In [50]:
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [112]:
(tree := DecisionTreeClassifier(class_weight = 'balanced')
)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the curre

In [54]:
grid_search_forest = GridSearchCV(forest, forest_grid, cv = folds, scoring='roc_auc', n_jobs=-1, verbose=1)

grid_search_forest.fit(X_diabetes, y_diabetes)

Fitting 5 folds for each of 108 candidates, totalling 540 fits


c:\Users\lr7273ow\AppData\Local\anaconda3\envs\polars\Lib\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestC...ht='balanced')
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'max_depth': [2, 3, ...], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 3, ...], 'n_estimators': [10, 100, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computat

In [102]:
(tree_grid :=
  { 'classifier': [DecisionTreeClassifier()], # Kept in a list to avoid that previous error!
    'classifier__min_samples_leaf': [1, 2, 3],
    'classifier__min_samples_split': [2, 3, 4],
    'classifier__max_depth': [3, 4, 5]
 }
)

{'classifier': [DecisionTreeClassifier()],
 'classifier__min_samples_leaf': [1, 2, 3],
 'classifier__min_samples_split': [2, 3, 4],
 'classifier__max_depth': [3, 4, 5]}

In [103]:
(forest_grid :=
{
    'classifier': [RandomForestClassifier()], # Added the 's' in classifier
    'classifier__max_depth': [2, 3, 4, 5],
    'classifier__min_samples_leaf': [1, 2, 3],
    'classifier__min_samples_split': [2, 3, 4],
    'classifier__n_estimators': [10, 100, 500]
}
)

{'classifier': [RandomForestClassifier()],
 'classifier__max_depth': [2, 3, 4, 5],
 'classifier__min_samples_leaf': [1, 2, 3],
 'classifier__min_samples_split': [2, 3, 4],
 'classifier__n_estimators': [10, 100, 500]}

In [104]:
(generic_cls :=
 Pipeline(steps = [
     ('scaler', StandardScaler()),
     ('classifier', None)  # <-- classifier goes here
 ])

)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [114]:
(combined_grid := 
 
 [forest_grid, tree_grid]
 

)

[{'classifier': [RandomForestClassifier()],
  'classifier__max_depth': [2, 3, 4, 5],
  'classifier__min_samples_leaf': [1, 2, 3],
  'classifier__min_samples_split': [2, 3, 4],
  'classifier__n_estimators': [10, 100, 500]},
 {'classifier': [DecisionTreeClassifier()],
  'classifier__min_samples_leaf': [1, 2, 3],
  'classifier__min_samples_split': [2, 3, 4],
  'classifier__max_depth': [3, 4, 5]}]

In [119]:
(combined_grid_search := 
 GridSearchCV(generic_cls,combined_grid, cv = folds, scoring='balanced_accuracy', verbose=1 )
 
 )

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...fier', None)])"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","[{'classifier': [RandomForestClassifier()], 'classifier__max_depth': [2, 3, ...], 'classifier__min_samples_leaf': [1, 2, ...], 'classifier__min_samples_split': [2, 3, ...], ...}, {'classifier': [DecisionTreeClassifier()], 'classifier__max_depth': [3, 4, ...], 'classifier__min_samples_leaf': [1, 2, ...], 'classifier__min_samples_split': [2, 3, ...]}]"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'balanced_accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strat

In [120]:
combined_grid_search = GridSearchCV(generic_cls, combined_grid, cv = folds, scoring='roc_auc', n_jobs=-1, verbose=1)

combined_grid_search.fit(X_train_diabetes, y_train_diabetes)

Fitting 5 folds for each of 135 candidates, totalling 675 fits


c:\Users\lr7273ow\AppData\Local\anaconda3\envs\polars\Lib\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...fier', None)])"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","[{'classifier': [RandomForestClassifier()], 'classifier__max_depth': [2, 3, ...], 'classifier__min_samples_leaf': [1, 2, ...], 'classifier__min_samples_split': [2, 3, ...], ...}, {'classifier': [DecisionTreeClassifier()], 'classifier__max_depth': [3, 4, ...], 'classifier__min_samples_leaf': [1, 2, ...], 'classifier__min_samples_split': [2, 3, ...]}]"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that c

In [118]:
grid_search_tree.best_params_

{'classifier': RandomForestClassifier(),
 'classifier__max_depth': 5,
 'classifier__min_samples_leaf': 2,
 'classifier__min_samples_split': 2,
 'classifier__n_estimators': 100}

In [60]:
(combined_grid := 
 
 [forest_grid, tree_grid]
 

)

[{'max_depth': [2, 3, 4, 5],
  'min_samples_leaf': [1, 2, 3],
  'min_samples_split': [2, 3, 4],
  'n_estimators': [10, 100, 500],
  'classifer': RandomForestClassifier()},
 {'min_samples_leaf': [1, 2, 3],
  'min_samples_split': [2, 3, 4],
  'max_depth': [3, 4, 5],
  'classifer': DecisionTreeClassifier()}]

In [121]:
my_scores = ['accuracy', 'balanced_accuracy', 'roc_auc']



(diabetes_scores :=  
 cross_validate(combined_grid_search, X_test_diabetes, y_test_diabetes,
                cv = folds,
                scoring=my_scores,
                verbose=0,
                n_jobs=-1,
                )
 >> rec.subset([f'test_{m}' for m in my_scores])
 >> rec.map(np.mean)
)



{'test_accuracy': np.float64(0.7145235892691952),
 'test_balanced_accuracy': np.float64(0.6374059139784947),
 'test_roc_auc': np.float64(nan)}

<font color="orange">
Your summary here.
</font>

### The best classifier out of the two options (random forest and decision tree) was the random forest classifier which was able to predict the classes with max depth of 5, classifier min sample leaf of 2, classifier min sample split of 2.  The cross validation map for our scores went as follows: 
### test_accuracy: 0.71 
### test_balanced_accuracy: 0.64
### test_roc_auc : NaN

## <font color="red"> Exercise 3 </font>

Now we build on the final exercise of the previous activity, where we performed a combined grid search over all of the classic classification methods.  Redo this, but this time add trees and forests of each type (alone, OvR, and OVO).

1. Create a combined grid on all 15 classifiers (5 classic classifiers + 5*OvR + 5*OvO).  
2. For `kNN`, add a number of `metric`s and `weights` to the grid.
3. Add 6 tree based classifiers to the grid search (tree + forest as well as OvR and OvO for each).
4. Perform the grid search over all the models and determing the winning model.
5. Evaluate the performance of the winning model using CV and write a summary interpreting each metric.

In [193]:
(knn_alone :=  
  {'classifier': [KNeighborsClassifier()],
   'classifier__n_neighbors': list(range(2,12,2)),
    'classifier__metric': ['l1','l2'],
    'classifier__weights': ['uniform','distance']
  }
 )

{'classifier': [KNeighborsClassifier()],
 'classifier__n_neighbors': [2, 4, 6, 8, 10],
 'classifier__metric': ['l1', 'l2'],
 'classifier__weights': ['uniform', 'distance']}

In [192]:
knn_wrapped = {
    'classifier': [
        OneVsRestClassifier(KNeighborsClassifier()), 
        OneVsOneClassifier(KNeighborsClassifier())
    ],
    'classifier__estimator__n_neighbors': list(range(2,12,2)),
    'classifier__estimator__metric': ['l1','l2'],
    'classifier__estimator__weights': ['uniform','distance']
}

In [ ]:
#(knn_wrapped := 
#    {'classifier': [OneVsOneClassifier(KNeighborsClassifier()),OneVsOneClassifier(KNeighborsClassifier())],
#     'classifier__estimator__n_neighbors': list(range(2,12,2)),
#     'classifier__estimator__metric': ['l1','l2'],
#     'classifier__estimator__weights': ['uniform','distance']
#    }
#)

{'classifier': [OneVsOneClassifier(estimator=KNeighborsClassifier()),
  OneVsOneClassifier(estimator=KNeighborsClassifier())],
 'classifier__estimator__n_neighbors': [2, 4, 6, 8, 10],
 'classifier__estimator__metric': ['l1', 'l2'],
 'classifier__estimator__weights': ['uniform', 'distance']}

In [194]:
(knn_grid :=
[knn_alone, knn_wrapped]
)

[{'classifier': [KNeighborsClassifier()],
  'classifier__n_neighbors': [2, 4, 6, 8, 10],
  'classifier__metric': ['l1', 'l2'],
  'classifier__weights': ['uniform', 'distance']},
 {'classifier': [OneVsRestClassifier(estimator=KNeighborsClassifier()),
   OneVsOneClassifier(estimator=KNeighborsClassifier())],
  'classifier__estimator__n_neighbors': [2, 4, 6, 8, 10],
  'classifier__estimator__metric': ['l1', 'l2'],
  'classifier__estimator__weights': ['uniform', 'distance']}]

In [181]:
# Your code here (add cells as needed.)


(nontuning_grid :=  {'classifier' :  
[LogisticRegression(max_iter=10000),
 OneVsRestClassifier(LogisticRegression(max_iter=10000)),
 OneVsOneClassifier(LogisticRegression(max_iter=10000)),
 GaussianNB(),
 OneVsRestClassifier(GaussianNB()),
 OneVsOneClassifier(GaussianNB()),
 LinearDiscriminantAnalysis(),
 OneVsRestClassifier(LinearDiscriminantAnalysis()),
 OneVsOneClassifier(LinearDiscriminantAnalysis()),
 QuadraticDiscriminantAnalysis(),
 OneVsRestClassifier(QuadraticDiscriminantAnalysis()),
 OneVsOneClassifier(QuadraticDiscriminantAnalysis())

]
}
)

{'classifier': [LogisticRegression(max_iter=10000),
  OneVsRestClassifier(estimator=LogisticRegression(max_iter=10000)),
  OneVsOneClassifier(estimator=LogisticRegression(max_iter=10000)),
  GaussianNB(),
  OneVsRestClassifier(estimator=GaussianNB()),
  OneVsOneClassifier(estimator=GaussianNB()),
  LinearDiscriminantAnalysis(),
  OneVsRestClassifier(estimator=LinearDiscriminantAnalysis()),
  OneVsOneClassifier(estimator=LinearDiscriminantAnalysis()),
  QuadraticDiscriminantAnalysis(),
  OneVsRestClassifier(estimator=QuadraticDiscriminantAnalysis()),
  OneVsOneClassifier(estimator=QuadraticDiscriminantAnalysis())]}

In [182]:
DecisionTreeClassifier().get_params()

{'ccp_alpha': 0.0,
 'class_weight': None,
 'criterion': 'gini',
 'max_depth': None,
 'max_features': None,
 'max_leaf_nodes': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'monotonic_cst': None,
 'random_state': None,
 'splitter': 'best'}

In [208]:
(tree_alone := 
 { 'classifier' : [DecisionTreeClassifier()],
    'classifier__max_depth': [3, 5, 10]

 }
 )

{'classifier': [DecisionTreeClassifier()], 'classifier__max_depth': [3, 5, 10]}

In [209]:
(tree_wrapped := 
 
 { 'classifier' : [OneVsRestClassifier(DecisionTreeClassifier()), 
                   OneVsOneClassifier(DecisionTreeClassifier())], 
    'classifier__estimator__max_depth': [3, 5, 10]                 
 }
 
)

{'classifier': [OneVsRestClassifier(estimator=DecisionTreeClassifier()),
  OneVsOneClassifier(estimator=DecisionTreeClassifier())],
 'classifier__estimator__max_depth': [3, 5, 10]}

In [210]:
(tree_grid :=
 
 [tree_alone,tree_wrapped]


)

[{'classifier': [DecisionTreeClassifier()],
  'classifier__max_depth': [3, 5, 10]},
 {'classifier': [OneVsRestClassifier(estimator=DecisionTreeClassifier()),
   OneVsOneClassifier(estimator=DecisionTreeClassifier())],
  'classifier__estimator__max_depth': [3, 5, 10]}]

In [211]:
(forest_alone := {
    'classifier': [RandomForestClassifier()],
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth': [5, 10]
}

)


{'classifier': [RandomForestClassifier()],
 'classifier__n_estimators': [50, 100],
 'classifier__max_depth': [5, 10]}

In [212]:
(forest_wrapped := {
    'classifier': [OneVsRestClassifier(RandomForestClassifier()), 
                    OneVsOneClassifier(RandomForestClassifier())],
    'classifier__estimator__n_estimators': [50, 100]
}

)

{'classifier': [OneVsRestClassifier(estimator=RandomForestClassifier()),
  OneVsOneClassifier(estimator=RandomForestClassifier())],
 'classifier__estimator__n_estimators': [50, 100]}

In [213]:
(forest_grid :=
 
    [forest_alone,forest_wrapped]

 )

[{'classifier': [RandomForestClassifier()],
  'classifier__n_estimators': [50, 100],
  'classifier__max_depth': [5, 10]},
 {'classifier': [OneVsRestClassifier(estimator=RandomForestClassifier()),
   OneVsOneClassifier(estimator=RandomForestClassifier())],
  'classifier__estimator__n_estimators': [50, 100]}]

In [ ]:
#(combined_grid := 

#[forest_grid,tree_grid,knn_grid,nontuning_grid]
#)

[[{'classifier': [RandomForestClassifier()],
   'classifier__n_estimators': [50, 100],
   'classifier__max_depth': [5, 10]},
  {'classifier': [OneVsRestClassifier(estimator=RandomForestClassifier()),
    OneVsOneClassifier(estimator=RandomForestClassifier())],
   'classifier__estimator__n_estimators': [50, 100]}],
 [{'classifier': [DecisionTreeClassifier()],
   'classifier__max_depth': [3, 5, 10]},
  {'classifier': [OneVsRestClassifier(estimator=DecisionTreeClassifier()),
    OneVsOneClassifier(estimator=DecisionTreeClassifier())],
   'classifier__estimator__max_depth': [3, 5, 10]}],
 [{'classifier': [KNeighborsClassifier()],
   'classifier__n_neighbors': [2, 4, 6, 8, 10],
   'classifier__metric': ['l1', 'l2'],
   'classifier__weights': ['uniform', 'distance']},
  {'classifier': [OneVsRestClassifier(estimator=KNeighborsClassifier()),
    OneVsOneClassifier(estimator=KNeighborsClassifier())],
   'classifier__estimator__n_neighbors': [2, 4, 6, 8, 10],
   'classifier__estimator__metric': 

In [215]:
(combined_grid := 
 
forest_grid + tree_grid + knn_grid + [nontuning_grid]
#[forest_grid,tree_grid,knn_grid,nontuning_grid]



)

[{'classifier': [RandomForestClassifier()],
  'classifier__n_estimators': [50, 100],
  'classifier__max_depth': [5, 10]},
 {'classifier': [OneVsRestClassifier(estimator=RandomForestClassifier()),
   OneVsOneClassifier(estimator=RandomForestClassifier())],
  'classifier__estimator__n_estimators': [50, 100]},
 {'classifier': [DecisionTreeClassifier()],
  'classifier__max_depth': [3, 5, 10]},
 {'classifier': [OneVsRestClassifier(estimator=DecisionTreeClassifier()),
   OneVsOneClassifier(estimator=DecisionTreeClassifier())],
  'classifier__estimator__max_depth': [3, 5, 10]},
 {'classifier': [KNeighborsClassifier()],
  'classifier__n_neighbors': [2, 4, 6, 8, 10],
  'classifier__metric': ['l1', 'l2'],
  'classifier__weights': ['uniform', 'distance']},
 {'classifier': [OneVsRestClassifier(estimator=KNeighborsClassifier()),
   OneVsOneClassifier(estimator=KNeighborsClassifier())],
  'classifier__estimator__n_neighbors': [2, 4, 6, 8, 10],
  'classifier__estimator__metric': ['l1', 'l2'],
  'clas

In [216]:
(combined_grid_search := 
GridSearchCV(generic_cls, combined_grid,cv=folds, scoring='balanced_accuracy', verbose=1)
)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...fier', None)])"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","[{'classifier': [RandomForestClassifier()], 'classifier__max_depth': [5, 10], 'classifier__n_estimators': [50, 100]}, {'classifier': [OneVsRestClas...tClassifier()), OneVsOneClass...tClassifier())], 'classifier__estimator__n_estimators': [50, 100]}, ...]"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'balanced_accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-

In [217]:
(combined_grid_search.fit(X_train_diabetes, y_train_diabetes)) 

Fitting 5 folds for each of 89 candidates, totalling 445 fits


c:\Users\lr7273ow\AppData\Local\anaconda3\envs\polars\Lib\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
c:\Users\lr7273ow\AppData\Local\anaconda3\envs\polars\Lib\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
c:\Users\lr7273ow\AppData\Local\anaconda3\envs\polars\Lib\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
c:\Users\lr7273ow\AppData\Local\anaconda3\envs\polars\Lib\site-packages\sklearn\base.py:1336: DataConversionW

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...fier', None)])"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","[{'classifier': [RandomForestClassifier()], 'classifier__max_depth': [5, 10], 'classifier__n_estimators': [50, 100]}, {'classifier': [OneVsRestClas...tClassifier()), OneVsOneClass...tClassifier())], 'classifier__estimator__n_estimators': [50, 100]}, ...]"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'balanced_accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-

In [218]:
print(f"Total models searched: {len(combined_grid_search.cv_results_['params'])}")
import pandas as pd
df = pd.DataFrame(combined_grid_search.cv_results_)
# This sorts them so you can see if QDA is at #2 or #20
print(df[['param_classifier', 'mean_test_score']].sort_values('mean_test_score', ascending=False))

Total models searched: 89
                                     param_classifier  mean_test_score
6   OneVsOneClassifier(estimator=RandomForestClass...         0.739050
78  OneVsRestClassifier(estimator=LogisticRegressi...         0.729634
79  OneVsOneClassifier(estimator=LogisticRegressio...         0.729634
77                 LogisticRegression(max_iter=10000)         0.729634
2                            RandomForestClassifier()         0.727250
..                                                ...              ...
27                             KNeighborsClassifier()         0.641027
47  OneVsRestClassifier(estimator=KNeighborsClassi...         0.641027
17                             KNeighborsClassifier()         0.639031
37  OneVsRestClassifier(estimator=KNeighborsClassi...         0.639031
57  OneVsOneClassifier(estimator=KNeighborsClassif...         0.639031

[89 rows x 2 columns]


In [220]:
metrics = ['accuracy',
           'balanced_accuracy',
           'f1_micro'
]


(all_combined_test_scores := 
 cross_validate(combined_grid_search, X_test_diabetes,y_test_diabetes,
                cv = folds, 
                scoring = metrics,
                verbose =1,
                n_jobs = -1
 ) 
  >>rec.subset([f'test_{m}' for m in metrics])
  >>rec.map(np.mean)
 )


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:  1.1min finished


{'test_accuracy': np.float64(0.757631822386679),
 'test_balanced_accuracy': np.float64(0.729502688172043),
 'test_f1_micro': np.float64(0.757631822386679)}

### Overall, the best classifier of the 89 models was the Random Forest Classifier with a test score of 0.74, the parameters of the classifier including a n_estimator of 50, criterion using 'gini', min samples split of 2 and a min samples leaf of 1. The metrics for the cross validation of the combined grid search went as follows: 


### 'test_accuracy':  0.757   
### 'test_balanced_accuracy' : 0.73  
###  'test_f1_micro' : 0.76
